In [51]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import numpy as np

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

url = "https://www.ambitionbox.com/list-of-companies?page=1"
driver.get(url)

# ✅ store HTML like requests
webpage = driver.page_source

print(webpage[:1000])   # print first part

<html lang="en" class=" " style="--vh: 6.2px; --dd-width: 351.4px;"><head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width,initial-scale=1,minimum-scale=1">
    <meta http-equiv="X-UA-Compatible" content="IE=edge"> 
    <link rel="manifest" href="/assets/next/manifest.json">
    <style>@media only screen and (min-width:767px){.trp-img{width:400px!important;max-width:400px!important}}</style>
    <script src="/static/js/env-runtime.js" defer=""></script>
    <script>window.dataLayer=window.dataLayer||[],window.gtag=window.gtag||function(){window.dataLayer.push(arguments)},gtag("js",new Date),window.initialDate=(new Date).toISOString()</script>
    <script>window.Prism=window.Prism||{},window.Prism.manual=!0</script>
    <title>Top Companies in India | AmbitionBox</title><meta data-n-head="ssr" name="copyright" content="2026 AmbitionBox"><meta data-n-head="ssr" name="revisit-after" content="1 day"><meta data-n-head="ssr" name="application-name" content="A

In [52]:
soup = BeautifulSoup(webpage, 'lxml')

In [53]:
companies = soup.find_all('div', class_='companyCardWrapper')
len(companies)

20

In [54]:
name = []
rating = []
review = []
salaries = []
interviews = []
jobs = []
benefits = []

for i in companies:
    # company name
    name.append(i.find('h2').text.strip())
    
    # rating
    rating.append(i.select_one('div.rating_text').text.strip())
    
    # review count
    review.append(i.find('span', class_='companyCardWrapper__companyRatingCount').text.strip())
    
    # all counts (important step)
    counts = i.find_all('span', class_='companyCardWrapper__ActionCount')
    
    # indexing based on UI order
    salaries.append(counts[1].text.strip() if len(counts) > 1 else None)
    interviews.append(counts[2].text.strip() if len(counts) > 2 else None)
    jobs.append(counts[3].text.strip() if len(counts) > 3 else None)
    benefits.append(counts[4].text.strip() if len(counts) > 4 else None)

In [56]:
df = pd.DataFrame({
    'Company': name,
    'Rating': rating,
    'Reviews': review,
    'Salaries': salaries,
    'Interviews': interviews,
    'Jobs': jobs,
    'Benefits': benefits,
})

df.head()

,Company,Rating,Reviews,Salaries,Interviews,Jobs,Benefits
0,TCS,3.3,(1.1L),10L,11.8k,3.6k,10.6k
1,Accenture,3.7,(72.6k),6.6L,9.3k,28.8k,6.7k
2,Wipro,3.6,(64.5k),4.8L,6.8k,394,4.6k
3,Cognizant,3.7,(60.8k),6L,6.4k,487,5.5k
4,Capgemini,3.7,(52.5k),4.8L,5.5k,2.6k,3.7k


In [57]:
df.shape

(20, 7)

In [58]:

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

final = []

for j in range(1, 11):

    url = f"https://www.ambitionbox.com/list-of-companies?page={j}"
    driver.get(url)
  

    webpage = driver.page_source
    soup = BeautifulSoup(webpage, 'html.parser')

    companies = soup.find_all('div', class_='companyCardWrapper')

    for i in companies:

        # name
        try:
            name = i.find('h2').text.strip()
        except:
            name = np.nan

        # rating
        try:
            rating = i.select_one('div.rating_text').text.strip()
        except:
            rating = np.nan

        # reviews
        try:
            reviews = i.find('span', class_='companyCardWrapper__companyRatingCount').text.strip()
        except:
            reviews = np.nan

        # counts (salaries, interviews, etc.)
        try:
            counts = i.find_all('span', class_='companyCardWrapper__ActionCount')

            salaries = counts[1].text.strip()
            interviews = counts[2].text.strip()
            jobs = counts[3].text.strip()
            benefits = counts[4].text.strip()

        except:
            salaries = np.nan
            interviews = np.nan
            jobs = np.nan
            benefits = np.nan

        # append row
        final.append({
            'Company': name,
            'Rating': rating,
            'Reviews': reviews,
            'Salaries': salaries,
            'Interviews': interviews,
            'Jobs': jobs,
            'Benefits': benefits
        })

# convert to DataFrame
df = pd.DataFrame(final)

# close browser
driver.quit()

df.head()

,Company,Rating,Reviews,Salaries,Interviews,Jobs,Benefits
0,TCS,3.3,(1.1L),10L,11.8k,3.6k,10.6k
1,Accenture,3.7,(72.6k),6.6L,9.3k,28.8k,6.7k
2,Wipro,3.6,(64.5k),4.8L,6.8k,394,4.6k
3,Cognizant,3.7,(60.8k),6L,6.4k,487,5.5k
4,Capgemini,3.7,(52.5k),4.8L,5.5k,2.6k,3.7k


In [62]:
df.shape

(200, 7)

In [63]:
df.to_csv('top_companies_india_dataset.csv', index = False)